<a href="https://colab.research.google.com/github/Venkatesh-Aleti-06/Learning-Notebooks/blob/main/Python/OOPS/SOLID/03_Liskov_Substitution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03. Liskov Substitution Principle (LSP)

## Definition
A **child class must work anywhere its parent works**, without breaking the program. The parent's methods are a promise, and every child must keep it.

## Real-world example
If every payment class inherits `refund()`, Cash on Delivery breaks the promise, because it can't refund online. Code that cancels orders crashes, and "if COD" checks spread everywhere.
Fix: split the promise. `Payment` has only `pay()`. `RefundablePayment` adds `refund()`. Card and UPI inherit `RefundablePayment`; COD inherits plain `Payment`.

## Key points
- Smell: an overridden method that **raises an error or does nothing**.
- Fix the **class hierarchy**, not the calling code.
- Liskov is about **keeping** the promise. Interface Segregation is about not making it **too big**.

## Interview line
"Liskov means any child can replace its parent without breaking things. If a child can't support a parent method, I split the hierarchy instead of throwing errors."

In [ ]:
from abc import ABC, abstractmethod

print("=" * 60)
print("   L - LISKOV: REAL-WORLD PAYMENT REFUND EXAMPLE")
print("=" * 60)

print("\n" + "#" * 60)
print("# BAD: every payment promises refund(), but Cash on Delivery can't")
print("#" * 60)

class PaymentBad(ABC):
    @abstractmethod
    def pay(self, amount): pass
    @abstractmethod
    def refund(self, amount): pass

class CardBad(PaymentBad):
    def pay(self, amount):    print(f"   [CARD] paid {amount}")
    def refund(self, amount): print(f"   [CARD] refunded {amount} to card")

class CODBad(PaymentBad):
    def pay(self, amount):    print(f"   [COD] collect {amount} at door")
    def refund(self, amount): raise Exception("COD cannot be refunded online!")

def cancel_order(payment, amount):
    print(f"[ORDER-SERVICE] cancelling, refunding via {type(payment).__name__}")
    try:
        payment.refund(amount)
    except Exception as e:
        print(f"   [CRASH] {e}")

cancel_order(CardBad(), 500)
cancel_order(CODBad(), 500)
print("[PROBLEM] OrderService trusted refund(). COD broke the promise.")
print("[PROBLEM] Now you add 'if type == COD' everywhere -> ugly.")

print("\n" + "#" * 60)
print("# GOOD: split the promise")
print("#" * 60)

class Payment(ABC):
    @abstractmethod
    def pay(self, amount): pass

class RefundablePayment(Payment):
    @abstractmethod
    def refund(self, amount): pass

class Card(RefundablePayment):
    def pay(self, amount):    print(f"   [CARD] paid {amount}")
    def refund(self, amount): print(f"   [CARD] refunded {amount} to card")

class UPI(RefundablePayment):
    def pay(self, amount):    print(f"   [UPI] paid {amount}")
    def refund(self, amount): print(f"   [UPI] refunded {amount} to UPI")

class COD(Payment):
    def pay(self, amount):    print(f"   [COD] collect {amount} at door")

def refund_order(payment: RefundablePayment, amount):
    print(f"[ORDER-SERVICE] refunding via {type(payment).__name__}")
    payment.refund(amount)

print(">>> All payments can pay:")
for p in [Card(), UPI(), COD()]:
    p.pay(500)

print("\n>>> Only RefundablePayments reach refund_order:")
for p in [Card(), UPI()]:
    refund_order(p, 500)

print("[RESULT] Any RefundablePayment works. No crash, no type checks.")

print("\n" + "=" * 60)
print("KEY TAKEAWAY")
print("=" * 60)
print("Parent = promise. Every child must keep it.")
print("If a child can't, move that method to a narrower parent.")

   L - LISKOV: REAL-WORLD PAYMENT REFUND EXAMPLE

############################################################
# BAD: every payment promises refund(), but Cash on Delivery can't
############################################################
[ORDER-SERVICE] cancelling, refunding via CardBad
   [CARD] refunded 500 to card
[ORDER-SERVICE] cancelling, refunding via CODBad
   [CRASH] COD cannot be refunded online!
[PROBLEM] OrderService trusted refund(). COD broke the promise.
[PROBLEM] Now you add 'if type == COD' everywhere -> ugly.

############################################################
# GOOD: split the promise
############################################################
>>> All payments can pay:
   [CARD] paid 500
   [UPI] paid 500
   [COD] collect 500 at door

>>> Only RefundablePayments reach refund_order:
[ORDER-SERVICE] refunding via Card
   [CARD] refunded 500 to card
[ORDER-SERVICE] refunding via UPI
   [UPI] refunded 500 to UPI
[RESULT] Any RefundablePayment works. No crash,